In [55]:
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *
from netgen.geom2d import SplineGeometry

a = 1 #m Länge
b = 2 #m Breite
 
shape = MoveTo(-1,-0.5).Rectangle(2,1).Face()
#circle = Circle((0,0),0.10).Face()
#circle.edges.name="circles"


shape.edges.name="free"
shape.edges.Min(X).name="dirichlet"
shape.edges.Max(X).name="dirichlet"


rect = shape    # -circle

geo = OCCGeometry(rect,dim=2)

# Draw(rect)
mesh = Mesh(geo.GenerateMesh(maxh=0.05))

mesh.Curve(3)
# Draw(mesh)

In [56]:
E  = 70e9      #Glas ~ 70 GPa Elastizitätsmodul
nu = 0.23
t  = 0.02     #Dicke [m]
rho = 2500   #kg/m^3 Dichte einer Aluminiumplatte
g = 9.81

q = rho * t * g     #Eigengewicht der Platte - rechte Seite der PDE

Db = E*t**3/(12*(1-nu**2))

def D(A):
    return Db *((1-nu)*A+ nu*Trace(A)*Id(2))

def Dinv(A):
    return 1/Db * (1/(1-nu)*A - nu/(1-nu**2)*Trace(A)*Id(2))



In [57]:
order = 3

V = HDivDiv(mesh, order=order-1,dirichlet="dirichlet")
Q = H1(mesh, order=order, dirichlet="dirichlet")
X = V * Q

(sigma,w),(tau,v)= X.TnT()

n = specialcf.normal(2)

def tang(u):
    return u - (u*n)*n

a = BilinearForm(X,symmetric=True)
a += InnerProduct(Dinv(sigma),tau) * dx
a += div(sigma)*Grad(v) * dx
a += div(tau) * Grad(w) * dx
a +=(-sigma[n,:] * tang(Grad(v)) - tau[n,:] * tang(Grad(w))  ) * dx(element_boundary=True)
a.Assemble()

L = LinearForm(X)
L += q * v *dx
L.Assemble()

gf_solution = GridFunction(X)
gf_solution.vec.data = a.mat.Inverse(X.FreeDofs(),inverse="") * L.vec

gf_sigma, gf_w = gf_solution.components
Draw(gf_sigma, mesh,name="sigma")
Draw(gf_w, mesh, name="disp",deformation=True, euler_angles=[-60,5,30])




WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'camera': {'euler_angles': […

BaseWebGuiScene